## Using relational model

In [1]:
import re

import polars as pl

from tool_extraction import SEED, artifact_siblings

artifact_siblings_with_content = (
    artifact_siblings
    .select("entry_sha", "content", "repo_full_name", "artifact_path", "entry_name", "content_fetched")
    .filter(pl.col("content_fetched") == 1)
    .with_columns(pl.col("content").str.strip_chars().replace("", None))
    .filter(pl.col("content").is_not_null())
    .drop("content_fetched")
)

# Every column gets the same seed so the sampled rows stay aligned.
sample = artifact_siblings_with_content.select(pl.all().sample(n=500, seed=SEED)).collect()

MAX_LENGTH = 384
MAX_WORDS_CONTEXT = 5

def get_snippet(text: str, entity: str) -> list[str]:
    pattern = re.compile(rf"(?<!\w){re.escape(entity)}(?!\w)")
    snippets = []
    for match in pattern.finditer(text):
        before = text[: match.start()].split()
        after = text[match.end() :].split()
        highlighted = f"[{match.group()}]"
        snippet = " ".join(before[-MAX_WORDS_CONTEXT:] + [highlighted] + after[:MAX_WORDS_CONTEXT])
        if len(before) > MAX_WORDS_CONTEXT:
            snippet = "..." + snippet
        if len(after) > MAX_WORDS_CONTEXT:
            snippet = snippet + "..."
        snippets.append(snippet)
    return snippets


In [2]:
from gliner import GLiNER
import polars as pl
import seaborn_polars as snl

relational_model = GLiNER.from_pretrained("knowledgator/gliner-relex-large-v0.5")

HIT_THRESHOLD = 0.6
HIT_RELATION_THRESHOLD = 0.6
HIT_LABELS = ["tool", "organization"]
HIT_RELATION_LABELS = ["from_organization"]
HIT_DTYPES = pl.Struct({
    "snippet": pl.List(pl.String),
    "entity": pl.String,
    "label": pl.String,
    "relation": pl.String,
    "related": pl.String,
    "related_label": pl.String,
    "score": pl.Float64,
})

def extract(text: str) -> list[dict]:
    clipped = (text or "")[:MAX_LENGTH]
    entities, relations = relational_model.inference(
        [clipped],
        labels=HIT_LABELS,
        relations=HIT_RELATION_LABELS,
        threshold=HIT_THRESHOLD,
        relation_threshold=HIT_RELATION_THRESHOLD,
    )
    return [
        {
            "entity": rel["head"]["text"],
            "label": rel["head"]["type"],
            "relation": rel["relation"],
            "related": rel["tail"]["text"],
            "related_label": rel["tail"]["type"],
            "score": rel["score"],
            "snippet": get_snippet(clipped, rel["head"]["text"]),
        }
        for rel in relations[0]
    ]

hits = (
    sample.lazy()
    .select("entry_sha", "content")
    .with_columns(
        pl.col("content")
        .map_elements(extract, return_dtype=pl.List(HIT_DTYPES))
        .alias("hits")
    )
    .explode("hits")
    .unnest("hits")
    .with_columns(
        pl.col("entity").str.to_lowercase(),
        pl.col("related").str.to_lowercase(),
        pl.when(pl.col("snippet").list.len() == 0)
        .then(pl.lit(["(no match)"]))
        .otherwise(pl.col("snippet"))
        .alias("snippet"),
    )
    .select(
        "entry_sha", "snippet", "entity", "label",
        "relation", "related", "related_label", "score",
    )
)

edges = (
    hits.group_by("entity", "label", "relation", "related", "related_label")
    .agg(
        pl.col("entry_sha").n_unique().alias("skills"),
        pl.col("score").mean().alias("score"),
    )
)

nodes = (
    pl.concat([
        hits.select(pl.col("entity").alias("name"), pl.col("label")),
        hits.select(pl.col("related").alias("name"), pl.col("related_label").alias("label")),
    ])
    .group_by("name")
    .agg(pl.col("label").mode().first().alias("label"))
)

review = hits.explode("snippet").head(20).collect()
with pl.Config(fmt_str_lengths=200, tbl_rows=20):
    display(review)

/home/nichmorgan/Projects/nau/msr-data-challenge-2027/scripts/.venv/lib/python3.13/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/home/nichmorgan/Projects/nau/msr-data-challenge-2027/scripts/.venv/lib/python3.13/site-packages/huggingface_hub/utils/_validators.py:190: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

entry_sha,snippet,entity,label,relation,related,related_label,score
str,str,str,str,str,str,str,f64
"""69301b4ae010905ede5a063048ca53ebcc7caabf""","""#!/usr/bin/env [python3] """"""Apply lifecycle edits to existing...""","""python3""","""tool""","""from_organization""","""word""","""tool""",0.828872
"""2b1ef3ef6ad9d6faa346bc02026a33b6a592dd24""","""#!/usr/bin/env [python3] """""" Validate MudBlazor CSS variables...""","""python3""","""tool""","""from_organization""","""dataclasse""","""organization""",0.887571
"""2e09f6a20cc44dfa35c8198f04d653f47fac397b""","""...tunnels.read description: Get a Cloudflare [Tunnel] by ID permission: tunnel.read request:...""","""tunnel""","""tool""","""from_organization""","""cloudflare""","""organization""",0.995089
"""2e09f6a20cc44dfa35c8198f04d653f47fac397b""","""...request: method: GET path: /accounts/{account_id}/cfd_tunnel/{ [tunnel_id] } inputs: - name: account_id...""","""tunnel_id""","""tool""","""from_organization""","""cloudflare""","""organization""",0.994947
"""2e09f6a20cc44dfa35c8198f04d653f47fac397b""","""...true location: path - name: [tunnel_id] type: string required: true location:...""","""tunnel_id""","""tool""","""from_organization""","""cloudflare""","""organization""",0.994947
"""e674f7086fa7426e60d8d3823c74ce6cfa83ccde""","""#!/bin/bash # [meeting-prep] /scripts/prep.sh — Research attendees for...""","""meeting-prep""","""organization""","""from_organization""","""euo""","""organization""",0.601902
"""e674f7086fa7426e60d8d3823c74ce6cfa83ccde""","""...CONFIG & ARGS --- PREP_DIR=""${PREP_DIR:-$HOME/.config/ [meeting-prep] }"" CONFIG_FILE=""$PREP_DIR/config.json"" HISTORY_FILE=""$PREP_DIR/brief-history.json"" LOG_FILE=""$PREP_DIR/prep-log.json"" BRIEFS_DI…","""meeting-prep""","""organization""","""from_organization""","""euo""","""organization""",0.601902
"""e674f7086fa7426e60d8d3823c74ce6cfa83ccde""","""...specific meeting (on-demand) set -euo [pipefail] # --- CONFIG & ARGS...""","""pipefail""","""tool""","""from_organization""","""meeting-prep""","""organization""",0.874535
"""9f0d13f8e9da916bdf7a896447656d08d58293e1""","""# [Chain Analysis] Use this when the user...""","""chain analysis""","""tool""","""from_organization""","""cha""","""organization""",0.95773
